# QUMO GPU benchmark on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ANFAIA/QUMO/blob/main/examples/colab/QUMO_GPU_Benchmark_Colab.ipynb)

This notebook downloads [QUMO](https://github.com/ANFAIA/QUMO) (a hardware-neutral quantum software
stack written in Mojo), builds it on the Colab machine and runs **one** benchmark with **one**
command: `bench/run_benchmarks.py`. Every state-vector engine runs the *same* layered circuit (10 layers
of `ry`+`rz` on every qubit plus a CX brick, the circuit of `bench/bench_sv.mojo`), on the same qubit
grid, with a warm-up and the median of several repetitions:

| engine | device | precision |
|---|---|---|
| QUMO (Mojo, MAX vendor-neutral GPU API) | CPU, GPU | CPU fp64; GPU fp32 and fp64 |
| Qiskit Aer | CPU, GPU (`qiskit-aer-gpu`) | fp64 (CPU); fp32 and fp64 (GPU) |
| NVIDIA CUDA-Q | GPU (`nvidia` target, cuStateVec) | fp32 and fp64 |
| PennyLane Lightning (optional) | GPU (`lightning.gpu`) | fp32 and fp64 |

It also cross-compiles QUMO's GPU kernels for this GPU's architecture, runs QUMO's GPU test suites on
the device and a small compiler benchmark (QUMO, Qiskit, TKET). Output: one JSON + CSV with the full
environment (GPU, driver, CPU, RAM, versions, commit, load average), tables and plots.

**Before you start:** *Runtime -> Change runtime type ->* a GPU (T4 on the free tier; L4, A100,
H100 or G4 on paid plans), then *Runtime -> Run all*. Rough estimate: 40-70 min on a T4 (the toolchain
download and the builds take 10-15 min of it); `QUICK` in step 1 is a smoke test of about 15 min.

**Trying several NVIDIA configurations:** tick `SAVE_TO_DRIVE` in step 7, run, then change the
runtime type (T4 -> L4 -> A100 ...) and *Run all* again. Step 8 merges every run saved in Drive into
one table and one plot. Changing the runtime type wipes `/content`, so save before switching.

**Hardware neutrality.** Nothing here is NVIDIA-specific except the NVIDIA-only baselines. Colab has
no AMD GPUs: use `QUMO_GPU_Benchmark_AMD.ipynb` (any Jupyter on a ROCm host) or `run_on_any_gpu.sh`,
with the same runner, circuits and output format. Method and sources:
[`docs/BENCHMARKING_ON_CLOUD_GPUS.md`](https://github.com/ANFAIA/QUMO/blob/main/docs/BENCHMARKING_ON_CLOUD_GPUS.md).

In [ ]:
#@title 0. Helpers (run first) { display-mode: "form" }
# Plain Python on purpose (no "!" shell magics), so every cell also runs in
# Jupyter / JupyterLab / VS Code and under nbclient.
import glob, json, os, re, shlex, shutil, subprocess, sys, time

REPO_URL = "https://github.com/ANFAIA/QUMO"


def sh(cmd, check=True, cwd=None):
    """Run a shell command, stream its output, return it; raise on failure if check."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace")
    out = []
    for line in p.stdout:
        if "Crashpad" not in line:
            print(line, end="", flush=True)
        out.append(line)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return "".join(out)


def find_checkout(start):
    """A QUMO checkout containing `start` (Makefile + qumo/ + bench/run_benchmarks.py), else None."""
    d = os.path.abspath(start)
    while True:
        if all(os.path.exists(os.path.join(d, x)) for x in ("Makefile", "qumo", "bench/run_benchmarks.py")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


def show_report(result_jsons, stem):
    """Tables + plots for result JSONs via examples/colab/compare_results.py, run in the
    benchmark venv (it has matplotlib), so the notebook kernel needs nothing extra."""
    from IPython.display import Image, Markdown, display
    md, png = stem + "_report.md", stem + "_time_vs_qubits.png"
    sh(f"bench/.venv/bin/python examples/colab/compare_results.py {' '.join(map(shlex.quote, result_jsons))} "
       f"--out {shlex.quote(md)} --plot {shlex.quote(png)}")
    with open(md) as fh:
        text = "\n".join(ln for ln in fh.read().splitlines() if not ln.startswith("!["))  # images shown below
    display(Markdown(text))
    for p in (png, png[:-4] + "_qumo_gpu.png"):
        if os.path.exists(p):
            display(Image(filename=p))
    return [p for p in (md, png, png[:-4] + "_qumo_gpu.png") if os.path.exists(p)]


IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
WORKDIR = "/content" if os.path.isdir("/content") else os.path.expanduser("~/qumo-bench")
EXISTING = find_checkout(os.getcwd())
REPO_DIR = EXISTING or os.path.join(WORKDIR, "QUMO")
print(f"Colab: {IN_COLAB}   repository directory: {REPO_DIR}" + ("  (existing checkout)" if EXISTING else ""))

## 1. GPU and configuration

`CONFIG = auto` lets the runner pick `bench/configs/<preset>.json` from the detected GPU (T4, L4, A100, H100/B200,
G4; any other NVIDIA GPU gets `nvidia_generic`, no GPU `cpu_only`); it is resolved in step 2, after the download.
Force a preset in the form if you like. `QUICK` runs tiny sizes with one repetition.

In [ ]:
#@title 1. Check the GPU { display-mode: "form" }
CONFIG = "auto"  #@param ["auto", "colab_t4", "colab_l4", "colab_a100", "colab_h100", "colab_g4", "nvidia_generic", "cpu_only", "quick"]
QUICK = False  #@param {type:"boolean"}


def query_nvidia():
    """[{name, memory_gib, driver, compute_cap, cuda}] from nvidia-smi; [] without an NVIDIA GPU."""
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=60)
        full = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60).stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return []
    m = re.search(r"CUDA Version:\s*([\d.]+)", full)
    gpus = []
    for line in r.stdout.strip().splitlines():
        f = [x.strip() for x in line.split(",")]
        if len(f) >= 4:
            gpus.append({"name": f[0], "memory_gib": float(f[1]) / 1024, "driver": f[2], "compute_cap": f[3],
                         "cuda": m.group(1) if m else None})
    return gpus


def driver_major(gpu):
    try:
        return int(gpu["driver"].split(".")[0])
    except (KeyError, ValueError, AttributeError):
        return None


GPUS = query_nvidia()
GPU = GPUS[0] if GPUS else None
if GPU:
    print(f"GPU: {GPU['name']}  {GPU['memory_gib']:.1f} GiB  driver {GPU['driver']}  "
          f"compute capability {GPU['compute_cap']}  (driver supports CUDA {GPU['cuda']})")
    try:
        if float(GPU["compute_cap"]) < 7.5:  # Mojo/MAX 26.6: Turing (sm_75) and newer
            print("NOTE: older than Turing (sm_75): the QUMO GPU suites will be skipped with a recorded reason; "
                  "QUMO CPU and the CUDA-Q / Aer GPU baselines still run.")
    except ValueError:
        pass
    if (driver_major(GPU) or 999) < 580:  # Mojo/MAX 26.6 wants driver >= 580, else a CUDA ptxas
        if os.path.exists("/usr/local/cuda/bin/ptxas"):
            os.environ["MODULAR_NVPTX_COMPILER_PATH"] = "/usr/local/cuda/bin/ptxas"
        print(f"NOTE: driver {GPU['driver']} < 580: MODULAR_NVPTX_COMPILER_PATH="
              f"{os.environ.get('MODULAR_NVPTX_COMPILER_PATH', '(no CUDA ptxas found)')}")
else:
    print("No NVIDIA GPU found (Runtime -> Change runtime type -> GPU). The run falls back to the CPU "
          "benchmarks and CPU baselines.")
print(f"requested configuration: {CONFIG}" + ("  (resolved in step 2)" if CONFIG == "auto" else "")
      + ("  --quick" if QUICK else ""))

## 2. Download QUMO

Default: latest `main`. Set `COMMIT` to benchmark an exact version.

In [ ]:
#@title 2. Download QUMO and resolve the preset { display-mode: "form" }
BRANCH = "main"  #@param {type:"string"}
COMMIT = ""  #@param {type:"string"}

if EXISTING:
    print(f"using the existing checkout {REPO_DIR} (BRANCH/COMMIT ignored)")
elif os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git -C {REPO_DIR} fetch --depth 1 origin {shlex.quote(COMMIT or BRANCH)}")
    sh(f"git -C {REPO_DIR} checkout --force FETCH_HEAD")
else:
    os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
    if COMMIT:
        sh(f"git clone --filter=blob:none {REPO_URL} {REPO_DIR}")
        sh(f"git -C {REPO_DIR} checkout {shlex.quote(COMMIT)}")
    else:
        sh(f"git clone --depth 1 --branch {shlex.quote(BRANCH)} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --format='%H %cd %s' --date=iso")
if not os.path.exists("bench/run_benchmarks.py"):
    raise RuntimeError("QM-E960: this QUMO version has no bench/run_benchmarks.py. Hint: use a newer BRANCH / COMMIT.")
# The runner picks the preset (one rule for every vendor; stdlib only, so the kernel's Python is enough).
sel = subprocess.run([sys.executable, "bench/run_benchmarks.py", "--select-config", "auto"],
                     capture_output=True, text=True)
lines = sel.stdout.splitlines()
if sel.returncode != 0 or not lines:
    raise RuntimeError("QM-E960: preset selection failed:\n" + sel.stdout + sel.stderr)
AUTO_CONFIG = lines[0].strip()
print("\n".join(lines[1:]))
CONFIG_NAME = AUTO_CONFIG if CONFIG == "auto" else CONFIG
CONFIG_PATH = f"bench/configs/{CONFIG_NAME}.json"
if not os.path.exists(CONFIG_PATH):
    raise RuntimeError(f"QM-E960: {CONFIG_PATH} not found in this QUMO version. Hint: pick a preset from bench/configs/.")
print(f"configuration: {CONFIG_NAME}" + ("  (auto)" if CONFIG == "auto" else f"  (forced; auto would be {AUTO_CONFIG})")
      + ("  --quick" if QUICK else ""))
print(json.load(open(CONFIG_PATH))["description"])


## 3. Install

Mojo + MAX from PyPI into `.venv` (Python 3.12, via `uv`; pinned to the versions QUMO is developed with, empty =
latest). Each GPU baseline gets its own virtual environment: CUDA-Q (`cuda-quantum-cu12` or `-cu13`, chosen from
the driver's CUDA version), `qiskit-aer-gpu` (its last release, 0.15.1, needs Qiskit < 2) and, optionally,
PennyLane `lightning.gpu`.

In [ ]:
#@title 3. Install the Mojo/MAX toolchain and the baselines { display-mode: "form" }
MOJO_VERSION = "1.1.0"  #@param {type:"string"}
MAX_VERSION = "26.6.0"  #@param {type:"string"}
INSTALL_CUDAQ = True  #@param {type:"boolean"}
INSTALL_AER_GPU = True  #@param {type:"boolean"}
INSTALL_LIGHTNING_GPU = False  #@param {type:"boolean"}
VENVS = os.path.join(WORKDIR, "venvs")

if not shutil.which("uv"):
    sh(f"{sys.executable} -m pip install -q uv")
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
# Mojo + MAX wheels (manylinux_2_34 x86_64 / aarch64: glibc >= 2.34) into ./.venv, pinned for reproducibility
if not os.path.exists(".venv/bin/mojo"):
    if MOJO_VERSION and MAX_VERSION:
        sh("uv venv -q .venv --python 3.12")
        sh(f"uv pip install -q --python .venv/bin/python mojo=={MOJO_VERSION} "
           f"max-mojo-libs=={MAX_VERSION} max-core=={MAX_VERSION}")
    else:
        sh("make setup")  # latest releases
sh(".venv/bin/mojo --version")
# Benchmark venv: QUMO's Python tooling + the CPU baselines (Aer CPU, TKET for the compiler subset, Stim, PyMatching)
if not os.path.exists("bench/.venv/bin/python"):
    sh("uv venv -q bench/.venv --python 3.12")
sh("uv pip install -q --python bench/.venv/bin/python qiskit qiskit-aer pytket pytket-qiskit stim pymatching "
   "numpy matplotlib psutil")


def baseline_venv(name, packages, env_var):
    """Own venv per GPU baseline (their CUDA / ROCm wheels and Qiskit pins must not clash).
    A failed install is reported and the suite records a skip; the notebook continues."""
    py = os.path.join(VENVS, name, "bin", "python")
    try:
        if not os.path.exists(py):
            sh(f"uv venv -q {os.path.join(VENVS, name)} --python 3.12")
        sh(f"uv pip install -q --python {py} {packages}")
        os.environ[env_var] = py
    except RuntimeError as e:
        print(f"NOTE: baseline '{name}' not installed ({e}); its suite will record a skip.")


if GPU:
    cuda_major = int(str(GPU.get("cuda") or "12").split(".")[0])
    if INSTALL_CUDAQ:  # wheels bundle their CUDA runtime; the driver must support that CUDA major version
        baseline_venv("cudaq", f"cuda-quantum-cu{13 if cuda_major >= 13 else 12} numpy", "QUMO_CUDAQ_PYTHON")
    if INSTALL_AER_GPU:  # Linux x86_64, CUDA 12 runtime from pip
        baseline_venv("aer-gpu", "'qiskit>=1.1,<2' qiskit-aer-gpu numpy", "QUMO_AER_GPU_PYTHON")
    if INSTALL_LIGHTNING_GPU:
        baseline_venv("lightning", "pennylane pennylane-lightning pennylane-lightning-gpu numpy", "QUMO_LIGHTNING_PYTHON")
print({k: v for k, v in os.environ.items() if k.startswith("QUMO_") or k == "MODULAR_NVPTX_COMPILER_PATH"})

## 4. Build

The runner compiles what the preset needs: `bench_sv` (CPU), the GPU benchmark twice (`-D QUMO_GPU` = fp64 and
`-D QUMO_GPU -D QUMO_GPU_FP32` = fp32) for this machine's GPU, and the Python extension for the compiler subset.
A failing build is printed here and recorded (not fatal) by the run.

In [ ]:
#@title 4. Build QUMO (benchmark binaries incl. GPU) { display-mode: "form" }
BUILD_CLI = False  #@param {type:"boolean"}

t0 = time.time()
if BUILD_CLI:
    sh("make cli-gpu" if GPU else "make cli")  # the qumo CLI + MCP server (not needed by the benchmark)
sh(f"bench/.venv/bin/python bench/run_benchmarks.py --config {CONFIG_PATH} --build-only", check=False)
print(f"build time: {time.time() - t0:.0f} s")

## 5. Run

`SUITES` (optional, comma separated) overrides the preset, e.g. `sv_gpu,baseline_cudaq`. Suites: `gpu_check,
sv_cpu, sv_gpu, baseline_aer, baseline_cudaq, baseline_lightning, neutral, qec, tests`. Each suite records `ok`,
`partial`, `skipped`, `failed` or `timeout` with a `QM-E96x` reason; nothing aborts the run.

In [ ]:
#@title 5. Run the benchmark { display-mode: "form" }
SUITES = ""  #@param {type:"string"}
RUN_LABEL = ""  #@param {type:"string"}

args = ["--config", CONFIG_PATH, "--out", "bench/results/cloud/"]
args += ["--quick"] if QUICK else []
args += ["--suites", SUITES] if SUITES else []
args += ["--label", RUN_LABEL] if RUN_LABEL else []
log = sh("bench/.venv/bin/python bench/run_benchmarks.py " + " ".join(shlex.quote(a) for a in args), check=False)
found = re.findall(r"^results: (\S+\.json)\s*$", log, re.M)
RESULT_JSON = found[-1] if found else max(glob.glob("bench/results/cloud/*.json"), key=os.path.getmtime)
print("result file:", RESULT_JSON)

## 6. Results

In [ ]:
#@title 6. Tables and plots { display-mode: "form" }
REPORT_FILES = show_report([RESULT_JSON], RESULT_JSON[:-5])

## 7. Save

In [ ]:
#@title 7. Save / download the results { display-mode: "form" }
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/QUMO_benchmarks"
import zipfile  # noqa: E402

stem = RESULT_JSON[:-5]
files_to_save = sorted(f for f in glob.glob(stem + "*") if os.path.isfile(f))
ZIP = os.path.join(WORKDIR, os.path.basename(stem) + ".zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_save:
        z.write(f, os.path.basename(f))
print("archive:", ZIP, "\n  " + "\n  ".join(os.path.basename(f) for f in files_to_save))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in files_to_save:
        shutil.copy(f, DRIVE_DIR)
    print("copied to", DRIVE_DIR)
try:
    from google.colab import files
    files.download(ZIP)
except ImportError:
    print("(not on Colab: download the archive with your Jupyter file browser)")

## 8. Compare NVIDIA configurations (optional)

Merge this run with earlier ones: the JSONs saved to Drive by step 7 (`FROM_DRIVE`) and/or files you upload (any
QUMO result JSON: another Colab GPU, an AMD or Apple run, your laptop). One table per machine, speed-ups, and two
plots: one panel per run, and the QUMO GPU series of every run on one axis.

In [ ]:
#@title 8. Compare with earlier runs { display-mode: "form" }
FROM_DRIVE = False  #@param {type:"boolean"}
UPLOAD = False  #@param {type:"boolean"}

paths = [RESULT_JSON] if "RESULT_JSON" in globals() else []
if FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    paths += sorted(glob.glob(os.path.join(DRIVE_DIR, "*.json")))
if UPLOAD:
    from google.colab import files
    up_dir = os.path.join(WORKDIR, "uploaded_results")
    os.makedirs(up_dir, exist_ok=True)
    for name, data in files.upload().items():
        with open(os.path.join(up_dir, os.path.basename(name)), "wb") as fh:
            fh.write(data)
        paths.append(os.path.join(up_dir, os.path.basename(name)))
unique = list({os.path.basename(p): p for p in paths if p.endswith(".json")}.values())
if len(unique) < 2:
    print("Need at least two result JSONs: tick FROM_DRIVE (runs saved by step 7) and/or UPLOAD.")
else:
    COMPARE_FILES = show_report(unique, os.path.join(WORKDIR, "qumo_compare"))
    print("comparison files:", COMPARE_FILES)

## Notes, unsupported GPUs and sharing

* **Supported NVIDIA GPUs** (Mojo/MAX 26.6): Turing (T4, `sm_75`) and newer: A100 (`sm_80`), A10 (`sm_86`),
  L4 (`sm_89`), H100/H200 (`sm_90`), B200 (`sm_100`), RTX PRO 6000 / RTX 50xx (`sm_120`). Driver >= 580; with an
  older driver step 1 sets `MODULAR_NVPTX_COMPILER_PATH=/usr/local/cuda/bin/ptxas` when that file exists. On an
  older GPU (K80, P100, V100) the QUMO GPU suites are **skipped with a recorded reason** and everything else
  still runs (QUMO CPU, Aer CPU/GPU, CUDA-Q).
* **fp64 on consumer / inference GPUs** (T4: 1/32 of fp32, L4 and RTX PRO 6000: 1/64) is slow for every
  engine: compare fp32 with fp32 and fp64 with fp64.
* **Memory:** widths are clamped to 60 % of GPU memory and of free host RAM (`memory_fraction` in the
  preset); every clamp is in the JSON. QUMO's GPU benchmark also runs the CPU reference at every width, so
  on a 12.7 GB free-tier runtime it stops at 26 qubits; *High-RAM* lifts that.
* **Failures never abort the run:** every suite records `ok` / `partial` / `skipped` / `failed` /
  `timeout` with a `QM-E96x` reason (`docs/ERRORS.md`); see *Problems and skipped parts* in step 6.
* **Share results:** attach the zip from step 7 to a GitHub issue, or open a pull request adding the JSON to
  `bench/results/cloud/`. The JSON stores no host names, user paths or credentials; keep `RUN_LABEL` free of
  personal data. This notebook needs no tokens.